In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import oracledb

<h1>모든 회원이 카테고리에 관계없이 무작위 기사를 읽은 경우</h1>

<h3> DB에서 로그테이블, 카테고리 선택 테이블, 뉴스테이블 획득 </h3>

In [71]:
# view_data = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\views.csv", encoding='cp949')
# log_data = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\log.csv", encoding='cp949')
# view_data = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\test2_v.csv", encoding='cp949')
log_data = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\test2_log1.csv", encoding='cp949', index_col=0)
pref_data = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\test2_preference.csv", encoding='cp949')
news_data = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\test2_news.csv", encoding='cp949')

# usr_article_views = pd.merge(log_data, view_data, on="title")
usr_article_views = pd.DataFrame(log_data)
usr_category = pd.DataFrame(pref_data)
news = pd.DataFrame(news_data)

# 테스트 코드이므로 임의의 로그, 선호도, 뉴스 csv 파일로 대체

In [32]:
# 회원별 선택한 카테고리 피벗 테이블화
usr_category_pivot = usr_category.pivot_table(index = 'mem_id', columns = 'cate_id', aggfunc = len, fill_value = 0)
usr_category_pivot

cate_id,IT/과학,경제,사회,생활/문화,세계,연예,정치
mem_id,,,,,,,
1,1,1,0,0,0,0,1
2,1,0,1,0,0,1,0
3,0,1,0,1,1,0,0
4,0,0,0,1,1,0,1
5,0,1,0,1,0,1,0
...,...,...,...,...,...,...,...
246,0,0,0,1,0,1,1
247,0,1,0,0,1,0,1
248,0,0,1,0,0,1,1


In [45]:
# 회원별 조회한 기사 피벗 테이블화
usr_article_pivot = usr_article_views.pivot_table(index='mem_id', columns=['cate_id', 'news_id'], aggfunc=len, fill_value=0)
usr_article_pivot

cate_id IT/과학                                               ...   정치       \
news_id  999  1000 1001 1002 1003 1004 1005 1006 1007 1008  ... 189  190    
mem_id                                                      ...             
1           0    0    0    0    0    0    0    0    0    0  ...    0    0   
2           0    1    0    0    0    0    0    0    0    0  ...    0    0   
3           0    0    0    0    0    0    0    0    0    0  ...    1    0   
4           0    0    0    0    0    0    0    0    0    0  ...    0    0   
5           0    0    0    0    0    0    0    0    2    0  ...    0    0   
...       ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
246         0    0    0    0    0    0    0    0    0    0  ...    0    0   
247         0    0    0    0    0    0    0    0    0    0  ...    0    0   
248         0    0    0    0    0    0    0    0    0    1  ...    0    0   
249         0    0    0    0    0    0    0    0    0    0  ...    0    0   
250         0    0    0    0    0    0    0    0    0    0  ...    0    0   

cate_id                                          
news_id 191  192  193  194  195  196  197  198   
mem_id                                           
1          0    0    0    0    0    0    0    0  
2          0    0    0    0    0    0    0    0  
3          0    0    0    0    0    0    0    0  
4          0    0    0    0    0    0    0    0  
5          0    0    0    0    0    1    0    0  
...      ...  ...  ...  ...  ...  ...  ...  ...  
246        0    0    0    0    0    0    0    1  
247        0    0    0    0    0    0    0    0  
248        0    0    0    0    1    1    0    0  
249        0    0    0    0    0    0    0    0  
250        0    0    0    0    0    0    0    0  

[250 rows x 1393 columns]

In [108]:
# 회원 간 유사도 판별
user_similarity = cosine_similarity(usr_article_pivot)
user_similarity

array([[1.        , 0.02778046, 0.08971226, ..., 0.04111132, 0.04450431,
        0.03090116],
       [0.02778046, 1.        , 0.0294916 , ..., 0.01351475, 0.04389043,
        0.03047492],
       [0.08971226, 0.0294916 , 1.        , ..., 0.10183502, 0.01574852,
        0.06560904],
       ...,
       [0.04111132, 0.01351475, 0.10183502, ..., 1.        , 0.08660254,
        0.0751646 ],
       [0.04450431, 0.04389043, 0.01574852, ..., 0.08660254, 1.        ,
        0.08136807],
       [0.03090116, 0.03047492, 0.06560904, ..., 0.0751646 , 0.08136807,
        1.        ]])

In [109]:
# def recommend(target): # target번 사용자에게 추천하는 경우
target = int(input("당신은 몇번 회원입니까?"))

# target 사용자와 유사한 사용자 탐색
similar_users = user_similarity[target - 1]
similar_users_indexes = similar_users.argsort()[::-1]

# 대상 사용자가 아직 시청하지 않은 기사 중에서 유사한 사용자가 시청한 기사 추천
recommended_articles = {}
flag = 0
for user_index in similar_users_indexes:
    if user_index == target - 1:
        continue  # 자기 자신은 제외
    selected_cate = [] # 선택한 카테고리 내
    non_selected_cate = [] # 선택하지 않은 카테고리 내
    score = 0
    for a_title in usr_article_pivot.columns:
        if usr_article_pivot.at[user_index + 1, a_title] == 1:
            # 겹칠 때마다 cnt ++ 다른 모든 기사에 점수 +cnt 
            if usr_article_pivot.at[target, a_title] == 1: # 겹치면 점수 +1
                score += 1
            else: # 안 겹치면 tmp에 추가
                t = list(a_title) # [카테고리 id, 기사 제목]
                if usr_category_pivot.at[target, t[0]] == 1: # target 유저의 카테고리 별 기사 분리
                    selected_cate.append(t[1])
                else:
                    non_selected_cate.append(t[1])

    for i in selected_cate:
        if i not in recommended_articles:
            recommended_articles[i] = score * 2  # 선택한 카테고리의 경우 점수 추가 부여
        else:
            recommended_articles[i] += score * 2

    for i in non_selected_cate:
        if i not in recommended_articles:
            recommended_articles[i] = score
        else:
            recommended_articles[i] += score
        

# 추후에 돌릴때는 기사 제목이 아닌 기사 번호로 추천해야함 -> 수행중

In [110]:
amount = 10 # 추천할 기사 갯수 설정
res = dict(sorted(recommended_articles.items(), key=lambda x : -x[1]))
print(list(res.keys())[:amount]) # 상위 점수 기사 id 10개

news_id = pd.DataFrame({"news_id" : res.keys()})
result_df = pd.merge(news_id, news, on="news_id")

[1003, 180, 390, 179, 132, 120, 1000, 1192, 1103, 1161]


In [151]:
print("" + str(target) + "번 회원님, 다음 기사를 추천합니다.")
print(f'{target}번 회원의 선택 카테고리\n', usr_category[usr_category['mem_id'] == target])
# print(res)
result_df.head(20)

1번 회원님, 다음 기사를 추천합니다.
1번 회원의 선택 카테고리
    mem_id cate_id
0       1   IT/과학
1       1      경제
2       1      정치


,news_id,cate_id,title
0,1003,IT/과학,"‘적자 늪’ 탈출한 CJ ENM, 작지만 소중한 74억원…“자회사 손실 줄어”"
1,180,정치,"尹 대통령, 새 대법원장 후보에 조희대 전 대법관 지명"
2,390,경제,영업익 나홀로 늘어난 SKT…‘5G 저가 요금제’ 출시 예고
3,179,정치,"이재명, 총선 인재 발굴·영입 나서...홍준표 ""이준석, 안 돌아올 것"""
4,132,정치,"“돈 많이 벌었으니 내라”…여야 다 꺼내든 이 세금, 어찌하오리까"
5,120,정치,"김건희 여사 ""소록도, 환자만의 공간 아냐…편견 없애려 왔어"""
6,1000,IT/과학,“이러니 다들 박은빈 타령” 결국 터졌다…추락한 엔터 명가 부활하나
7,1192,IT/과학,"‘게임은 잘 나가는데’… 컴투스, 미디어사업 부진에 3분기 적자"
8,1103,IT/과학,"최주희 티빙 ""가입자 연내 400만·BEP 내년 하반기 달성"" 전망"
9,1161,IT/과학,"넷마블, 신작 ‘아스달 연대기: 세 개의 세력’ 타이틀명 확정…티저 공개"


In [134]:
print("" + str(target) + "번 회원과 유사도가 높은 회원 목록(내림차순)")
for i in similar_users_indexes:
    if i + 1 == target:continue
    print(i + 1, end=" ")

1번 회원과 유사도가 높은 회원 목록(내림차순)
89 77 79 198 201 180 210 22 148 38 149 67 229 3 135 93 237 49 96 11 147 186 194 21 212 43 137 222 181 57 100 200 134 157 136 177 151 126 56 125 8 118 70 150 9 142 162 20 233 59 82 18 144 152 224 217 48 50 42 129 207 203 28 115 103 23 83 107 61 62 175 123 164 240 178 206 36 169 76 130 205 131 235 245 214 4 33 63 114 220 190 81 163 41 40 156 241 15 209 238 53 60 172 218 16 171 141 86 108 159 102 167 44 84 161 138 239 29 119 199 191 154 249 128 51 213 97 52 75 10 78 185 94 166 98 143 176 27 85 74 170 64 228 37 13 69 242 66 25 232 248 26 80 120 202 195 231 230 92 139 187 112 95 219 189 179 34 243 173 188 6 146 46 250 87 31 133 234 91 73 71 35 109 90 111 127 24 140 226 106 47 72 122 244 110 183 168 158 2 116 182 215 5 184 160 165 225 7 193 55 30 227 196 32 208 192 105 247 117 124 104 39 221 153 45 19 145 68 223 211 88 65 246 216 17 204 197 99 101 54 12 58 14 113 174 121 132 155 236 

In [143]:
# 유사도 체크
a = 1
b = 89
asdf = f'{a} 번 회원과 {b} 번 회원의'
print(asdf, "유사도 : ", user_similarity[a-1][b-1]) # 유사도
print(f'{a}번 회원의 선택 카테고리 : ', usr_category[usr_category['mem_id'] == a])
print(f'{b}번 회원의 선택 카테고리 : ', usr_category[usr_category['mem_id'] == b])

1 번 회원과 89 번 회원의 유사도 :  0.15859036992388778
1번 회원의 선택 카테고리 :     mem_id cate_id
0       1   IT/과학
1       1      경제
2       1      정치
89번 회원의 선택 카테고리 :       mem_id cate_id
264      89      세계
265      89      사회
266      89      연예


전부 난수로 기사를 배치했기 때문에 유사도가 높은 회원이라 할지라도 2가지 문제 생김<br>
1. 선택 카테고리가 완전히 다르지만 읽은 기사 종류가 같아지는 문제<br>
2. 선택 카테고리가 아님에도 점수가 지나치게 높으면 추천항목에 뜰 수 있는 문제<br>

해결 방안<br>
1 -> 유저가 고른 카테고리는 난수처리 하지만, 유저가 읽은 기사는 고른 카테고리 내에서만 읽게 변경<br>
2 -> 선택한 카테고리와 선택하지 않은 카테고리의 딕셔너리를 분리<br>